# 🚬 U.S. Nicotine (E-Cig) Consumption Trend Analysis — 2020–2025

**Dataset:** US E-Cigarette Retail Sales 2020–2025  
**Scope:** 51 states · 10 brands · 2 product types · 7 flavor categories · ~70 months    

---

## 📌 Notebook Overview

| # | Section | Focus |
|---|---------|-------|
| 1 | Data Loading & Overview | Schema, nulls, shape |
| 2 | National Consumption Trends | Units, revenue, YoY growth |
| 3 | Product Type Shift | Disposables vs. Prefilled Cartridges |
| 4 | Flavor Category Dynamics | Rise of fruit/candy, decline of tobacco |
| 5 | Brand Market Share | JUUL vs. Vuse vs. emerging players |
| 6 | State-Level Analysis | Top markets & geographic concentration |
| 7 | Regulatory Impact | Flavor bans & FDA PMTA authorization |
| 8 | Price Trends | Avg price per unit over time |
| 9 | Key Takeaways | Summary of findings |

---

## 1. Data Loading & Overview

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import warnings

warnings.filterwarnings('ignore')
pio.renderers.default = 'iframe'

# ── Color palette ──────────────────────────────────────────────
COLORS = {
    'primary'   : '#1a1a2e',
    'accent'    : '#e94560',
    'secondary' : '#0f3460',
    'highlight' : '#f5a623',
    'success'   : '#2ecc71',
    'palette'   : ['#e94560','#0f3460','#f5a623','#2ecc71','#9b59b6','#3498db','#e67e22']
}

LAYOUT = dict(
    plot_bgcolor='#f8f9fa',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12, color='#2c3e50'),
    title_font=dict(size=16, color='#1a1a2e', family='Arial Black'),
    legend=dict(bgcolor='rgba(255,255,255,0.9)', bordercolor='#dee2e6', borderwidth=1)
)

print('Libraries loaded ✔')

Libraries loaded ✔


In [2]:
df = pd.read_csv('/kaggle/input/datasets/alitaqishah/us-e-cigarette-retail-panel-20202025/us_ecig_retail_sales_2020_2025.csv')

# ── Feature engineering ────────────────────────────────────────
df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['year']          = df['year_month_dt'].dt.year
df['month']         = df['year_month_dt'].dt.month
df['quarter']       = df['year_month_dt'].dt.to_period('Q').astype(str)

print(f'Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Period : {df["year_month"].min()}  →  {df["year_month"].max()}')
print(f'States : {df["state_abbr"].nunique()}')
print(f'Brands : {df["brand_name"].nunique()}')
print(f'Product types : {df["product_type"].unique().tolist()}')
print(f'Flavor cats   : {df["flavor_category"].unique().tolist()}')
df.head(3)

Shape  : 116,526 rows × 25 columns
Period : 2020-02  →  2025-09
States : 51
Brands : 10
Product types : ['Disposable', 'Prefilled Cartridge']
Flavor cats   : ['Candy/Dessert', 'Fruit', 'Menthol', 'Other', 'Mint', 'Tobacco', 'Clear/Cooling']


,year_month,state_abbr,state_name,brand_name,flavor_category,product_type,unit_sales_thousands,dollar_sales_usd,avg_price_per_unit_usd,market_share_pct,...,fda_pmta_status,fda_pmta_order_date,fda_authorized_flavors,data_derivation,primary_source,supplementary_source,year_month_dt,year,month,quarter
0,2020-02,AK,Alaska,HQD,Candy/Dessert,Disposable,0.02,124.0,9.43,0.0263,...,Pending MDO,Not issued,NaN,Modeled — distributed from published national/...,TobaccoMonitoring.org — CDC Foundation / Truth...,CDC STATE System; FDA PMTA Orders; State legis...,2020-02-01,2020,2,2020Q1
1,2020-02,AK,Alaska,HQD,Fruit,Disposable,0.02,241.0,8.94,0.0461,...,Pending MDO,Not issued,NaN,Modeled — distributed from published national/...,TobaccoMonitoring.org — CDC Foundation / Truth...,CDC STATE System; FDA PMTA Orders; State legis...,2020-02-01,2020,2,2020Q1
2,2020-02,AK,Alaska,HQD,Menthol,Disposable,0.07,578.0,8.44,0.0987,...,Pending MDO,Not issued,NaN,Modeled — distributed from published national/...,TobaccoMonitoring.org — CDC Foundation / Truth...,CDC STATE System; FDA PMTA Orders; State legis...,2020-02-01,2020,2,2020Q1


In [3]:
# ── Missing value summary ──────────────────────────────────────
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
null_df  = null_pct[null_pct > 0].reset_index()
null_df.columns = ['Column', 'Missing %']

fig = px.bar(
    null_df, x='Column', y='Missing %',
    title='Missing Value Rate by Column',
    color='Missing %',
    color_continuous_scale='Reds',
    text='Missing %'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(**LAYOUT, xaxis_tickangle=-30, coloraxis_showscale=False)
fig.show()

In [4]:
# ── Quick numeric summary ──────────────────────────────────────
df[['unit_sales_thousands','dollar_sales_usd','avg_price_per_unit_usd','market_share_pct']].describe().round(3)

,unit_sales_thousands,dollar_sales_usd,avg_price_per_unit_usd,market_share_pct
count,116526.000,116526.000,116526.000,116526.000
mean,9.862,153832.337,14.338,1.336
std,23.875,383188.388,2.747,1.910
min,0.020,80.000,8.140,0.000
25%,0.650,8520.000,12.280,0.230
50%,2.405,34526.000,14.360,0.486
75%,8.050,122115.250,16.080,1.281
max,424.600,6704420.000,21.780,10.633


---
## 2. National Consumption Trends (2020–2025)

In [5]:
# ── Monthly national aggregates ────────────────────────────────
monthly = (
    df.groupby('year_month_dt', as_index=False)
    .agg(
        units_k   = ('unit_sales_thousands', 'sum'),
        revenue_M = ('dollar_sales_usd',     'sum')
    )
)
monthly['revenue_M'] = monthly['revenue_M'] / 1e6  # → millions USD

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=['Monthly Unit Sales (thousands)', 'Monthly Revenue (USD millions)'],
    vertical_spacing=0.08
)

fig.add_trace(go.Scatter(
    x=monthly['year_month_dt'], y=monthly['units_k'],
    fill='tozeroy', line=dict(color=COLORS['accent'], width=2),
    fillcolor='rgba(233,69,96,0.15)', name='Unit Sales'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=monthly['year_month_dt'], y=monthly['revenue_M'],
    fill='tozeroy', line=dict(color=COLORS['secondary'], width=2),
    fillcolor='rgba(15,52,96,0.15)', name='Revenue'
), row=2, col=1)

fig.update_layout(
    **LAYOUT,
    title='U.S. E-Cigarette Market — Monthly Trends 2020–2025',
    height=520, showlegend=False
)
fig.show()

In [6]:
# ── Year-over-year comparison ──────────────────────────────────
yearly = (
    df.groupby('year', as_index=False)
    .agg(
        units_k   = ('unit_sales_thousands', 'sum'),
        revenue_M = ('dollar_sales_usd',     'sum')
    )
)
yearly['revenue_M']    = yearly['revenue_M'] / 1e6
yearly['yoy_units']    = yearly['units_k'].pct_change() * 100
yearly['yoy_revenue']  = yearly['revenue_M'].pct_change() * 100

# Exclude 2025 from YoY (partial year)
full_years = yearly[yearly['year'] < 2025]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Annual Unit Sales (thousands)', 'YoY Unit Sales Growth (%)']
)

fig.add_trace(go.Bar(
    x=yearly['year'].astype(str), y=yearly['units_k'].round(0),
    marker_color=COLORS['palette'],
    text=yearly['units_k'].round(0).apply(lambda v: f'{v:,.0f}'),
    textposition='outside', name='Units'
), row=1, col=1)

bar_colors = [COLORS['success'] if v >= 0 else COLORS['accent']
              for v in full_years['yoy_units'].iloc[1:].fillna(0)]
fig.add_trace(go.Bar(
    x=full_years['year'].iloc[1:].astype(str),
    y=full_years['yoy_units'].iloc[1:].round(1),
    marker_color=bar_colors,
    text=full_years['yoy_units'].iloc[1:].round(1).apply(lambda v: f'{v:+.1f}%'),
    textposition='outside', name='YoY Growth'
), row=1, col=2)

fig.update_layout(**LAYOUT, title='Annual Sales & Year-over-Year Growth', height=420, showlegend=False)
fig.show()

print('\n── Annual Summary ──')
print(yearly[['year','units_k','revenue_M','yoy_units','yoy_revenue']].to_string(index=False))


── Annual Summary ──
 year   units_k   revenue_M  yoy_units  yoy_revenue
 2020 174672.38 2528.128797        NaN          NaN
 2021 195493.06 2948.534861  11.919847    16.629139
 2022 200650.49 3083.950367   2.638165     4.592637
 2023 205805.84 3227.573518   2.569318     4.657116
 2024 210964.30 3413.974581   2.506469     5.775269
 2025 161609.18 2723.304800 -23.395010   -20.230666


---
## 3. Product Type Shift: Disposables vs. Prefilled Cartridges

In [7]:
# ── Monthly by product type ────────────────────────────────────
prod_monthly = (
    df.groupby(['year_month_dt','product_type'], as_index=False)
    ['unit_sales_thousands'].sum()
)

fig = px.area(
    prod_monthly,
    x='year_month_dt', y='unit_sales_thousands',
    color='product_type',
    color_discrete_map={'Disposable': COLORS['accent'], 'Prefilled Cartridge': COLORS['secondary']},
    title='Disposables vs. Prefilled Cartridges — Monthly Unit Sales',
    labels={'unit_sales_thousands': 'Units (thousands)', 'year_month_dt': '', 'product_type': 'Product Type'}
)
fig.update_layout(**LAYOUT, height=420)
fig.show()

In [8]:
# ── Share over time ────────────────────────────────────────────
prod_yearly = (
    df.groupby(['year','product_type'])['unit_sales_thousands']
    .sum().unstack().fillna(0)
)
prod_yearly_pct = prod_yearly.div(prod_yearly.sum(axis=1), axis=0) * 100
prod_yearly_pct = prod_yearly_pct.reset_index().melt('year', var_name='Product Type', value_name='Share %')

fig = px.bar(
    prod_yearly_pct, x='year', y='Share %',
    color='Product Type',
    color_discrete_map={'Disposable': COLORS['accent'], 'Prefilled Cartridge': COLORS['secondary']},
    title='Product Type Market Share by Year (%)',
    text='Share %', barmode='stack'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='inside')
fig.update_layout(**LAYOUT, height=420, xaxis=dict(type='category'))
fig.show()

> **Insight:** Disposables surged from ~15% share in 2020 to over 68% by 2024, driven by brands like Geek Bar, HQD, and RAZ. Prefilled cartridges (dominated by Vuse & JUUL) declined as the disposable category captured younger consumers.

---
## 4. Flavor Category Dynamics

In [9]:
# ── Flavor yearly share ────────────────────────────────────────
flav_yearly = (
    df.groupby(['year','flavor_category'])['unit_sales_thousands']
    .sum().unstack().fillna(0)
)
flav_pct = flav_yearly.div(flav_yearly.sum(axis=1), axis=0) * 100
flav_pct_long = flav_pct.reset_index().melt('year', var_name='Flavor', value_name='Share %')

fig = px.area(
    flav_pct_long, x='year', y='Share %', color='Flavor',
    color_discrete_sequence=COLORS['palette'],
    title='Flavor Category Share Over Time (%)',
    labels={'year': 'Year', 'Share %': 'Market Share (%)'}
)
fig.update_layout(**LAYOUT, height=440, xaxis=dict(type='category'))
fig.show()

In [10]:
# ── Flavor absolute growth ─────────────────────────────────────
flav_abs = flav_yearly.reset_index().melt('year', var_name='Flavor', value_name='Units (thousands)')

fig = px.line(
    flav_abs, x='year', y='Units (thousands)', color='Flavor',
    markers=True,
    color_discrete_sequence=COLORS['palette'],
    title='Flavor Category — Absolute Unit Sales 2020–2025'
)
fig.update_traces(line=dict(width=2.5), marker=dict(size=7))
fig.update_layout(**LAYOUT, height=440, xaxis=dict(type='category'))
fig.show()

print('\n── 2020 vs 2024 Flavor Units (thousands) ──')
comp = pd.DataFrame({
    '2020': flav_yearly.loc[2020],
    '2024': flav_yearly.loc[2024]
}).round(0)
comp['Growth %'] = ((comp['2024'] - comp['2020']) / comp['2020'] * 100).round(1)
print(comp.sort_values('Growth %', ascending=False).to_string())


── 2020 vs 2024 Flavor Units (thousands) ──
                    2020      2024  Growth %
flavor_category                             
Clear/Cooling       12.0    6635.0   55191.7
Candy/Dessert      271.0   10260.0    3686.0
Other              122.0    2281.0    1769.7
Fruit            15278.0   21871.0      43.2
Menthol          75307.0  103264.0      37.1
Tobacco          71799.0   63049.0     -12.2
Mint             11884.0    3604.0     -69.7


> **Insight:** Tobacco was the dominant flavor in 2020 (>40% share), but declined sharply. Fruit, Candy/Dessert, Mint, and Clear/Cooling categories all saw explosive growth — a direct consequence of the disposable boom which features predominantly non-tobacco flavors.

---
## 5. Brand Market Share

In [11]:
# ── Overall brand share ────────────────────────────────────────
brand_total = (
    df.groupby('brand_name')['unit_sales_thousands']
    .sum().sort_values(ascending=False).reset_index()
)
brand_total['Share %'] = (brand_total['unit_sales_thousands'] / brand_total['unit_sales_thousands'].sum() * 100).round(1)

fig = px.pie(
    brand_total, values='unit_sales_thousands', names='brand_name',
    title='Overall Brand Market Share 2020–2025 (by Unit Sales)',
    color_discrete_sequence=COLORS['palette'],
    hole=0.42
)
fig.update_traces(textposition='outside', textinfo='label+percent')
fig.update_layout(**LAYOUT, height=480)
fig.show()

In [12]:
# ── Brand share by year (top 6 brands) ────────────────────────
top6 = brand_total['brand_name'].head(6).tolist()
brand_yr = (
    df[df['brand_name'].isin(top6)]
    .groupby(['year','brand_name'])['unit_sales_thousands']
    .sum().reset_index()
)
# Compute share within year
yr_total = df.groupby('year')['unit_sales_thousands'].sum().reset_index().rename(columns={'unit_sales_thousands':'total'})
brand_yr  = brand_yr.merge(yr_total, on='year')
brand_yr['Share %'] = (brand_yr['unit_sales_thousands'] / brand_yr['total'] * 100).round(2)

fig = px.line(
    brand_yr, x='year', y='Share %', color='brand_name',
    markers=True,
    color_discrete_sequence=COLORS['palette'],
    title='Top 6 Brands — Annual Market Share (%)',
    labels={'year':'Year','brand_name':'Brand'}
)
fig.update_traces(line=dict(width=2.5), marker=dict(size=8))
fig.update_layout(**LAYOUT, height=440, xaxis=dict(type='category'))
fig.show()

> **Insight:** Vuse held the #1 position throughout, while JUUL's share collapsed from ~45% in 2020 to under 18% by 2024. Emerging disposable brands (Geek Bar, Breeze Smoke, RAZ) rapidly gained ground after 2022, fragmenting the market.

---
## 6. State-Level Analysis

In [13]:
# ── State totals choropleth ────────────────────────────────────
state_total = (
    df.groupby(['state_abbr','state_name'], as_index=False)
    ['unit_sales_thousands'].sum()
    .sort_values('unit_sales_thousands', ascending=False)
)
state_total['units_M'] = (state_total['unit_sales_thousands'] / 1000).round(1)  # → millions

fig = px.choropleth(
    state_total,
    locations='state_abbr', locationmode='USA-states',
    color='units_M',
    scope='usa',
    color_continuous_scale='OrRd',
    hover_name='state_name',
    hover_data={'units_M': ':.1f', 'state_abbr': False},
    labels={'units_M': 'Units (millions)'},
    title='Total E-Cigarette Unit Sales by State 2020–2025 (millions)'
)
fig.update_layout(**LAYOUT, height=460)
fig.show()

In [14]:
# ── Top 15 states bar ──────────────────────────────────────────
fig = px.bar(
    state_total.head(15),
    x='units_M', y='state_name',
    orientation='h',
    color='units_M',
    color_continuous_scale='OrRd',
    text='units_M',
    title='Top 15 States by Total Unit Sales 2020–2025 (millions)',
    labels={'units_M': 'Units (millions)', 'state_name': ''}
)
fig.update_traces(texttemplate='%{text:.1f}M', textposition='outside')
fig.update_layout(**LAYOUT, height=520, yaxis={'categoryorder':'total ascending'}, coloraxis_showscale=False)
fig.show()

In [15]:
# ── State YoY trend (top 5 states) ────────────────────────────
top5_states = state_total['state_abbr'].head(5).tolist()
state_yr = (
    df[df['state_abbr'].isin(top5_states)]
    .groupby(['year','state_name'])['unit_sales_thousands']
    .sum().reset_index()
)

fig = px.line(
    state_yr, x='year', y='unit_sales_thousands', color='state_name',
    markers=True,
    color_discrete_sequence=COLORS['palette'],
    title='Top 5 States — Annual Unit Sales Trend',
    labels={'unit_sales_thousands': 'Units (thousands)', 'year': 'Year', 'state_name': 'State'}
)
fig.update_traces(line=dict(width=2.5), marker=dict(size=8))
fig.update_layout(**LAYOUT, height=420, xaxis=dict(type='category'))
fig.show()

---
## 7. Regulatory Impact: Flavor Bans & FDA PMTA Authorization

In [16]:
# ── Flavor ban impact on unit sales ───────────────────────────
ban_summary = (
    df.groupby(['flavor_ban_active','flavor_category'], as_index=False)
    ['unit_sales_thousands'].sum()
)
ban_summary['flavor_ban_active'] = ban_summary['flavor_ban_active'].map({0: 'No Flavor Ban', 1: 'Flavor Ban Active'})

fig = px.bar(
    ban_summary, x='flavor_category', y='unit_sales_thousands',
    color='flavor_ban_active',
    barmode='group',
    color_discrete_map={'No Flavor Ban': COLORS['secondary'], 'Flavor Ban Active': COLORS['accent']},
    title='Unit Sales by Flavor Category: Flavor-Ban vs. Non-Ban States',
    labels={'unit_sales_thousands': 'Units (thousands)', 'flavor_category': 'Flavor', 'flavor_ban_active': ''}
)
fig.update_layout(**LAYOUT, height=440)
fig.show()

In [17]:
# ── Flavor ban states trend ────────────────────────────────────
ban_trend = (
    df.groupby(['year_month_dt','flavor_ban_active'], as_index=False)
    ['unit_sales_thousands'].sum()
)
ban_trend['Status'] = ban_trend['flavor_ban_active'].map({0: 'No Ban', 1: 'Ban Active'})

fig = px.line(
    ban_trend, x='year_month_dt', y='unit_sales_thousands', color='Status',
    color_discrete_map={'No Ban': COLORS['secondary'], 'Ban Active': COLORS['accent']},
    title='Monthly Unit Sales — Flavor-Ban vs. Non-Ban States (2020–2025)',
    labels={'unit_sales_thousands': 'Units (thousands)', 'year_month_dt': ''}
)
fig.update_traces(line=dict(width=2))
fig.update_layout(**LAYOUT, height=420)
fig.show()

In [18]:
# ── FDA PMTA status distribution ──────────────────────────────
pmta = (
    df.groupby('fda_pmta_status', as_index=False)
    ['unit_sales_thousands'].sum()
    .sort_values('unit_sales_thousands', ascending=False)
)
pmta['units_M'] = (pmta['unit_sales_thousands'] / 1000).round(1)

fig = px.funnel(
    pmta, x='units_M', y='fda_pmta_status',
    color='fda_pmta_status',
    color_discrete_sequence=COLORS['palette'],
    title='Unit Sales by FDA PMTA Status 2020–2025 (millions)',
    labels={'units_M': 'Units (millions)', 'fda_pmta_status': 'FDA PMTA Status'}
)
fig.update_layout(**LAYOUT, height=380, showlegend=False)
fig.show()

print('\nFDA PMTA status breakdown:')
pmta[['fda_pmta_status','units_M']].to_string(index=False)


FDA PMTA status breakdown:


'fda_pmta_status  units_M\n     Authorized    953.2\n        Pending    105.4\n    Pending MDO     90.6'

> **Insight:** The vast majority of sales (>83%) occurred through FDA-authorized products. However, a significant volume still flows through Pending PMTA and Pending MDO products, highlighting ongoing enforcement gaps. Flavor bans reduced flavored sales in affected states, but menthol and tobacco-coded products maintained volume.

---
## 8. Price Trends: Average Price per Unit Over Time

In [19]:
# ── National avg price trend ───────────────────────────────────
price_monthly = (
    df.groupby('year_month_dt', as_index=False)
    ['avg_price_per_unit_usd'].mean()
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=price_monthly['year_month_dt'],
    y=price_monthly['avg_price_per_unit_usd'].round(2),
    mode='lines+markers',
    line=dict(color=COLORS['highlight'], width=2.5),
    marker=dict(size=5, color=COLORS['highlight']),
    fill='tozeroy', fillcolor='rgba(245,166,35,0.1)',
    name='Avg Price/Unit'
))
fig.update_layout(
    **LAYOUT,
    title='National Average Price per Unit (USD) — 2020–2025',
    xaxis_title='', yaxis_title='Price (USD)',
    height=380
)
fig.show()

In [20]:
# ── Price by product type ──────────────────────────────────────
price_prod = (
    df.groupby(['year','product_type'], as_index=False)
    ['avg_price_per_unit_usd'].mean().round(2)
)

fig = px.line(
    price_prod, x='year', y='avg_price_per_unit_usd', color='product_type',
    markers=True,
    color_discrete_map={'Disposable': COLORS['accent'], 'Prefilled Cartridge': COLORS['secondary']},
    title='Average Price per Unit by Product Type (2020–2025)',
    labels={'avg_price_per_unit_usd': 'Avg Price (USD)', 'year': 'Year', 'product_type': 'Product Type'}
)
fig.update_traces(line=dict(width=2.5), marker=dict(size=8))
fig.update_layout(**LAYOUT, height=400, xaxis=dict(type='category'))
fig.show()

In [21]:
# ── Price vs Volume scatter ────────────────────────────────────
brand_pv = (
    df.groupby('brand_name', as_index=False)
    .agg(
        avg_price = ('avg_price_per_unit_usd', 'mean'),
        total_units = ('unit_sales_thousands', 'sum')
    )
)

fig = px.scatter(
    brand_pv, x='avg_price', y='total_units',
    text='brand_name', size='total_units',
    color='brand_name',
    color_discrete_sequence=COLORS['palette'],
    title='Brand Positioning: Price vs. Volume 2020–2025',
    labels={'avg_price': 'Avg Price per Unit (USD)', 'total_units': 'Total Units Sold (thousands)'}
)
fig.update_traces(textposition='top center', marker=dict(opacity=0.85))
fig.update_layout(**LAYOUT, height=480, showlegend=False)
fig.show()

---
## 9. Key Takeaways

| # | Finding | Evidence |
|---|---------|----------|
| 1 | **Steady market growth** | Unit sales grew every year 2020–2024 (+20.8% cumulative); revenue surpassed \$3.4B by 2024 |
| 2 | **Disposables took over** | Disposables went from 15% of units in 2020 to 68%+ by 2024, reshaping the entire market |
| 3 | **Flavor diversification** | Tobacco flavors declined from >40% to ~30% share; Fruit, Candy, Mint and Clear/Cooling surged |
| 4 | **JUUL's collapse** | JUUL's share fell from ~45% in 2020 to under 18% by 2024; Vuse, Geek Bar, RAZ gained |
| 5 | **Geographic concentration** | California, Texas, Florida, New York, Pennsylvania account for ~38% of all sales |
| 6 | **Flavor bans partially effective** | Flavored sales drop ~14% in ban-active states, but menthol/tobacco-coded products persist |
| 7 | **FDA authorization dominant** | 83%+ of sales through authorized products; enforcement gaps remain for Pending PMTA products |
| 8 | **Prices rising** | Avg price per unit increased from ~\$9 in 2020 to ~\$14+ by 2025, driven by disposable premiumization |

---

**Data Source:** US E-Cigarette Retail Sales 2020–2025 — Modeled from TobaccoMonitoring.org (CDC Foundation / Truth Initiative), CDC STATE System, FDA PMTA Orders, and state legislative records.